In [6]:
import os

print(os.listdir("/content"))

['.config', 'medical_training (1).jslon', 'sample_data']


In [7]:
!pip install datasets

In [8]:
import numpy as np
import pandas as pd
import json
import re


Data Exploration


In [9]:
from datasets import load_dataset

In [10]:
dataset=load_dataset("lavita/MedQuAD")

README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…): reconstructing file:   0%|          |  0.00B / 10.7MB            

data/train-00000-of-00001-e36383d177026d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

In [11]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})


In [12]:
dataset.keys()

dict_keys(['train'])

In [13]:
train_data=dataset['train']

In [14]:
len(train_data)

47441

In [15]:
train_data.column_names

['document_id',
 'document_source',
 'document_url',
 'category',
 'umls_cui',
 'umls_semantic_types',
 'umls_semantic_group',
 'synonyms',
 'question_id',
 'question_focus',
 'question_type',
 'question',
 'answer']

In [16]:
train_data.features

{'document_id': Value('string'),
 'document_source': Value('string'),
 'document_url': Value('string'),
 'category': Value('string'),
 'umls_cui': Value('string'),
 'umls_semantic_types': Value('string'),
 'umls_semantic_group': Value('string'),
 'synonyms': Value('string'),
 'question_id': Value('string'),
 'question_focus': Value('string'),
 'question_type': Value('string'),
 'question': Value('string'),
 'answer': Value('string')}

In [17]:
train_data[0]

{'document_id': '0000559',
 'document_source': 'GHR',
 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair',
 'category': None,
 'umls_cui': 'C0343073',
 'umls_semantic_types': 'T047',
 'umls_semantic_group': 'Disorders',
 'synonyms': 'KWWH',
 'question_id': '0000559-1',
 'question_focus': 'keratoderma with woolly hair',
 'question_type': 'information',
 'question': 'What is (are) keratoderma with woolly hair ?',
 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the

In [18]:
for i in range(5):
  print(train_data[i])

{'document_id': '0000559', 'document_source': 'GHR', 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair', 'category': None, 'umls_cui': 'C0343073', 'umls_semantic_types': 'T047', 'umls_semantic_group': 'Disorders', 'synonyms': 'KWWH', 'question_id': '0000559-1', 'question_focus': 'keratoderma with woolly hair', 'question_type': 'information', 'question': 'What is (are) keratoderma with woolly hair ?', 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of th

In [19]:
df=train_data.to_pandas()

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47441 entries, 0 to 47440
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   document_id          47436 non-null  object
 1   document_source      47441 non-null  object
 2   document_url         47441 non-null  object
 3   category             32010 non-null  object
 4   umls_cui             31417 non-null  object
 5   umls_semantic_types  31375 non-null  object
 6   umls_semantic_group  31417 non-null  object
 7   synonyms             24669 non-null  object
 8   question_id          47441 non-null  object
 9   question_focus       47427 non-null  object
 10  question_type        47441 non-null  object
 11  question             47441 non-null  object
 12  answer               16407 non-null  object
dtypes: object(13)
memory usage: 4.7+ MB


In [21]:
df.shape

(47441, 13)

In [22]:
df.isnull().sum()

,0
document_id,5
document_source,0
document_url,0
category,15431
umls_cui,16024
umls_semantic_types,16066
umls_semantic_group,16024
synonyms,22772
question_id,0
question_focus,14


In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
df['question'].str.len().describe()

,question
count,47441.000000
mean,51.537531
std,15.771672
min,14.000000
25%,40.000000
50%,51.000000
75%,62.000000
max,191.000000


In [25]:
df['answer'].str.len().describe()

,answer
count,16407.000000
mean,1303.452673
std,1656.694326
min,6.000000
25%,487.000000
50%,890.000000
75%,1589.000000
max,29046.000000


In [26]:
df.to_csv("medquad_raw.csv",index=False)

Data Preprocessing


In [27]:
df=pd.read_csv("medquad_raw.csv")

In [28]:
df.columns.tolist()

['document_id',
 'document_source',
 'document_url',
 'category',
 'umls_cui',
 'umls_semantic_types',
 'umls_semantic_group',
 'synonyms',
 'question_id',
 'question_focus',
 'question_type',
 'question',
 'answer']

In [29]:
# remove missing question/answer
df = df.dropna(subset=["question", "answer"])

print("Shape after removing missing values:", df.shape)

Shape after removing missing values: (16407, 13)


In [30]:
# clean whitespace
df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

In [31]:
before = len(df)

df = df.drop_duplicates(
    subset=["question", "answer"]
).reset_index(drop=True)

after = len(df)

print("Duplicates removed:", before - after)
print("Final records:", after)

Duplicates removed: 48
Final records: 16359


In [32]:
print(df.head(5))

  document_id document_source  \
0     0000559             GHR   
1     0000559             GHR   
2     0000559             GHR   
3     0000559             GHR   
4     0000559             GHR   

                                        document_url category  umls_cui  \
0  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
1  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
2  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
3  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
4  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   

  umls_semantic_types umls_semantic_group synonyms question_id  \
0                T047           Disorders     KWWH   0000559-1   
1                T047           Disorders     KWWH   0000559-2   
2                T047           Disorders     KWWH   0000559-3   
3                T047           Disorders     KWWH   0000559-4   
4                T04

In [33]:
def create_instruction(row):
  return {
       "instruction": "Answer the following medical question accurately and clearly.",
        "input": row["question"],
        "output": row["answer"]
  }

In [34]:
processed_data=df.apply(create_instruction,axis=1).tolist()

In [35]:
print(processed_data[0])

{'instruction': 'Answer the following medical question accurately and clearly.', 'input': 'What is (are) keratoderma with woolly hair ?', 'output': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appe

In [36]:
def create_training_text(row):
    return (
        "### Instruction:\n"
        "Answer the following medical question accurately and clearly.\n\n"
        "### Question:\n"
        f"{row['question']}\n\n"
        "### Answer:\n"
        f"{row['answer']}"
    )

In [37]:
df["text"]=df.apply(create_training_text,axis=1)

In [38]:
print(df["text"][0])

### Instruction:
Answer the following medical question accurately and clearly.

### Question:
What is (are) keratoderma with woolly hair ?

### Answer:
Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not a

In [39]:
print("Final dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Final dataset shape: (16359, 14)

Columns:
['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer', 'text']


In [40]:
df[['question','answer','text']].to_csv("processed.csv",index=False)

In [41]:
with open(
    "medical_training.jslon",
    "w",
    encoding="utf-8"
)as file:
  for item in processed_data:
    file.write(json.dumps(item,ensure_ascii=False)+"\n")


In [42]:
with open(
    "medical_training.jslon",
    "r",
    encoding="utf-8"
) as f:
    first_line = f.readline()

print(first_line)

{"instruction": "Answer the following medical question accurately and clearly.", "input": "What is (are) keratoderma with woolly hair ?", "output": "Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appe

In [43]:
print("Final number of records:", len(df))

print("\nMissing values:")
print(df[["question", "answer", "text"]].isnull().sum())

print("\nDuplicate records:")
print(df[["question", "answer"]].duplicated().sum())

Final number of records: 16359

Missing values:
question    0
answer      0
text        0
dtype: int64

Duplicate records:
0


Base Model Selection & Loading

In [44]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")


PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [45]:
!pip install -q transformers accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM

In [46]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Model:", MODEL_NAME)

Model: Qwen/Qwen2.5-1.5B-Instruct


In [47]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
print("tokenizer loaded successfully")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

tokenizer loaded successfully


In [48]:
text="what is diabates"
tokens=tokenizer(text)
print(tokens)


{'input_ids': [12555, 374, 1853, 370, 973], 'attention_mask': [1, 1, 1, 1, 1]}


In [49]:
print("Input_ids")
print(tokens['input_ids'])

Input_ids
[12555, 374, 1853, 370, 973]


In [50]:
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16

)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [51]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Model loaded on:", device)

Model loaded on: cuda


In [52]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(
    "Model type:",
    model.config.model_type
)

print(
    "Hidden size:",
    model.config.hidden_size
)

print(
    "Number of layers:",
    model.config.num_hidden_layers
)

print(
    "Vocabulary size:",
    model.config.vocab_size
)
if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated() / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved() / 1024**3
    )

    print("\nGPU Memory")
    print(
        f"Allocated: {allocated:.2f} GB"
    )

    print(
        f"Reserved: {reserved:.2f} GB"
    )

Total parameters: 1,543,714,304
Model type: qwen2
Hidden size: 1536
Number of layers: 28
Vocabulary size: 151936

GPU Memory
Allocated: 2.88 GB
Reserved: 3.06 GB


In [53]:
messages = [
    {
        "role": "user",
        "content": "What is diabetes?"
    }
]

# Convert conversation into model's chat format
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\n" + "=" * 60)
print("BASE MODEL TEST")
print("=" * 60)

# Tokenize prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

# Generate response
with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

# Remove original prompt tokens
input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

# Convert tokens back to text
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("\nQuestion:")
print("What is diabetes?")

print("\nBase Model Response:")
print(response)



BASE MODEL TEST

Question:
What is diabetes?

Base Model Response:
Diabetes is a chronic metabolic disorder characterized by high blood sugar levels due to insulin resistance or insufficient production of the hormone insulin by the pancreas. There are two main types: Type 1 Diabetes and Type 2 Diabetes.

- **Type 1 Diabetes**: This occurs when the body's immune system attacks and destroys the insulin-producing cells in the pancreas (pancreatic beta cells), leading to little or no insulin production. It can develop at any age but often appears during childhood or adolescence.


In [54]:
print(" Final Verification ")
print("✓ Base model:", MODEL_NAME)
print("✓ Tokenizer loaded:", tokenizer is not None)
print("✓ Model loaded:", model is not None)
print("✓ Device:", device)

if torch.cuda.is_available():
    print(
        "✓ GPU:",
        torch.cuda.get_device_name(0)
    )

 Final Verification 
✓ Base model: Qwen/Qwen2.5-1.5B-Instruct
✓ Tokenizer loaded: True
✓ Model loaded: True
✓ Device: cuda
✓ GPU: Tesla T4


QLoRA CONFIGURATION


In [55]:
!pip install -q -U transformers accelerate bitsandbytes peft

In [56]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

In [57]:
print("check GPU")
print("CUDA Available :", torch.cuda.is_available())
if not torch.cuda.is_available():
   raise RuntimeError(
        "GPU not available. Go to Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name())
MODEL_NAME
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nTokenizer loaded successfully!")

check GPU
CUDA Available : True
GPU: Tesla T4

Tokenizer loaded successfully!


In [58]:
#Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\n4-bit Quantization Configuration:")
print(bnb_config)
#Load Base Model in 4-bit
print("\nLoading model in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("4-bit model loaded successfully!")



4-bit Quantization Configuration:
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


Loading model in 4-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit model loaded successfully!


In [59]:
!pip install -U peft

In [60]:
# LoRA Configuration

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("\nLoRA configuration created:")
print(lora_config)

# Apply Lora to Model

model = get_peft_model(
    model,
    lora_config
)

print("\nLoRA adapter applied successfully!")


LoRA configuration created:
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'k_proj', 'q_proj', 'v_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)

LoRA adapter a

In [61]:
print("TRAINABLE PARAMETERS")


model.print_trainable_parameters()

print("Model Device",model.device)
print(torch.cuda.get_device_name(0))


if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated() / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved() / 1024**3
    )

    print("\nGPU Memory:")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")


TRAINABLE PARAMETERS
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
Model Device cuda:0
Tesla T4

GPU Memory:
Allocated: 1.13 GB
Reserved: 4.10 GB


In [62]:
df['text'].head(2)


,text
0,### Instruction:\nAnswer the following medical...
1,### Instruction:\nAnswer the following medical...


QLoRA Fine Tunning

In [63]:
# ============================================================
# ACTUAL QLoRA FINE-TUNING
# ============================================================

# Install required libraries
!pip install -q -U trl datasets transformers accelerate bitsandbytes peft


# ============================================================
#  Imports
# ============================================================

import os
import torch

from datasets import load_dataset

from transformers import (
    TrainingArguments,
    BitsAndBytesConfig
)

from trl import SFTConfig, SFTTrainer


# ============================================================
# GPU CHECK
# ============================================================

print("=" * 60)
print("GPU CHECK")
print("=" * 60)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. Go to Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print(f"Total GPU Memory: {gpu_memory:.2f} GB")


# ============================================================
# DATASET PATH
# ============================================================

DATA_PATH = "/content/medical_training (1).jslon"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

print("\nDataset found:")
print(DATA_PATH)


# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset(
    "json",
    data_files=DATA_PATH,
    split="train"
)

dataset = dataset.shuffle(seed=42).select(range(5000))

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Number of records:", len(dataset))
print("Columns:", dataset.column_names)


# ============================================================
# VERIFY DATASET FORMAT
# ============================================================

required_columns = [
    "instruction",
    "input",
    "output"
]

for column in required_columns:
    if column not in dataset.column_names:
        raise ValueError(
            f"Missing required column: {column}"
        )

print("\nDataset format is correct!")


# ============================================================
# . CREATE TRAINING TEXT
# ============================================================

def create_training_text(example):

    instruction = str(example["instruction"]).strip()
    question = str(example["input"]).strip()
    answer = str(example["output"]).strip()

    return {
        "text": (
            "### Instruction:\n"
            + instruction
            + "\n\n"
            "### Question:\n"
            + question
            + "\n\n"
            "### Answer:\n"
            + answer
        )
    }


dataset = dataset.map(
    create_training_text
)


# ============================================================
#  CHECK TRAINING EXAMPLE
# ============================================================

print("\n" + "=" * 60)
print("TRAINING EXAMPLE")
print("=" * 60)

print(dataset[0]["text"])


# ============================================================
#  CHECK DATASET
# ============================================================

print("\nDataset columns after formatting:")
print(dataset.column_names)

print("\nTotal training examples:")
print(len(dataset))


# ============================================================
#  TOKENIZER
# ============================================================

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nTokenizer ready.")
print("Pad token:", tokenizer.pad_token)


# ============================================================
#  TRAINING OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = "./medical-qlora-output"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
#  TRAINING CONFIGURATION
# ============================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    # Number of epochs
    num_train_epochs=1,

    # Batch size
    per_device_train_batch_size=4,

    # Gradient accumulation
    gradient_accumulation_steps=2,

    # Learning rate
    learning_rate=2e-4,

    # Optimizer
    optim="paged_adamw_8bit",



    # Maximum sequence length
    max_length=384,

    # Column containing training text
    dataset_text_field="text",






    # Logging
    logging_steps=10,

    # Save checkpoints
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # Reproducibility
    seed=42,

    # Disable external logging
    report_to="none",

    # Memory optimization
    gradient_checkpointing=False
)

print("\nTraining configuration created!")


# ============================================================
# . CREATE SFT TRAINER
# ============================================================

trainer = SFTTrainer(

    model=model,

    args=training_args,

    train_dataset=dataset,

    processing_class=tokenizer
)

print("\nSFTTrainer created successfully!")


# ============================================================
#  TRAINABLE PARAMETERS
# ============================================================

print("\n" + "=" * 60)
print("TRAINABLE PARAMETERS")
print("=" * 60)

model.print_trainable_parameters()


# ============================================================
#  START ACTUAL TRAINING
# ============================================================

print("\n" + "=" * 60)
print("STARTING MEDICAL FINE-TUNING")
print("=" * 60)

print("Training started...")
print("Do not disconnect the Colab runtime.")

train_result = trainer.train()


# ============================================================
# . TRAINING COMPLETED
# ============================================================

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print(train_result)


# ============================================================
#  TRAINING METRICS
# ============================================================

metrics = train_result.metrics

print("\nTraining Metrics:")

for key, value in metrics.items():
    print(f"{key}: {value}")


# ============================================================
#  SAVE TRAINER STATE
# ============================================================

trainer.save_state()

print("\nTrainer state saved.")


# ============================================================
#  SAVE LORA ADAPTER
# ============================================================

ADAPTER_PATH = "./medical-qlora-adapter"

model.save_pretrained(
    ADAPTER_PATH
)

tokenizer.save_pretrained(
    ADAPTER_PATH
)

print("\nLoRA adapter saved at:")
print(ADAPTER_PATH)


# ============================================================
#  SHOW SAVED FILES
# ============================================================

print("\n" + "=" * 60)
print("SAVED ADAPTER FILES")
print("=" * 60)

for file in os.listdir(ADAPTER_PATH):
    print(file)


# ============================================================
# GPU MEMORY
# ============================================================

allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

reserved = (
    torch.cuda.memory_reserved()
    / 1024**3
)

print("\nGPU Memory:")
print(f"Allocated: {allocated:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")


# ============================================================
# 22. FINAL VERIFICATION
# ============================================================


print("\nAdapter location:")
print(ADAPTER_PATH)

# ============================================================
# DOWNLOAD TRAINED LORA ADAPTER
# ============================================================

import shutil
from google.colab import files

ZIP_PATH = shutil.make_archive(
    "/content/medical-qlora-adapter",
    "zip",
    "/content/medical-qlora-adapter"
)

print("=" * 60)
print("ADAPTER ZIP CREATED")
print("=" * 60)

print(ZIP_PATH)

# Download to computer
files.download(ZIP_PATH)



GPU CHECK
GPU: Tesla T4
Total GPU Memory: 14.56 GB

Dataset found:
/content/medical_training (1).jslon


Generating train split: 0 examples [00:00, ? examples/s]


DATASET INFORMATION
Number of records: 5000
Columns: ['instruction', 'input', 'output']

Dataset format is correct!


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]


TRAINING EXAMPLE
### Instruction:
Answer the following medical question accurately and clearly.

### Question:
What are the symptoms of Congenital generalized lipodystrophy type 2 ?

### Answer:
What are the signs and symptoms of Congenital generalized lipodystrophy type 2? The Human Phenotype Ontology provides the following list of signs and symptoms for Congenital generalized lipodystrophy type 2. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Acanthosis nigricans - Accelerated skeletal maturation - Acute pancreatitis - Autosomal recessive inheritance - Cirrhosis - Clitoromegaly - Congenital onset - Cystic angiomatosis of bone - Decreased fertility - Decreased fertility in females - Decreased serum leptin - Generalized muscular appearance from birth

...

Step,Training Loss
10,1.724249
20,1.497267
30,1.374306
40,1.354457
50,1.199084
60,1.265558
70,1.281515
80,1.093511
90,1.217858
100,1.107702


Step,Training Loss
10,1.724249
20,1.497267
30,1.374306
40,1.354457
50,1.199084
60,1.265558
70,1.281515
80,1.093511
90,1.217858
100,1.107702



TRAINING COMPLETED
TrainOutput(global_step=625, training_loss=1.1578949737548827, metrics={'train_runtime': 6025.6809, 'train_samples_per_second': 0.83, 'train_steps_per_second': 0.104, 'total_flos': 1.3760667862253568e+16, 'train_loss': 1.1578949737548827, 'entropy': 1.0946276247501374, 'num_tokens': 1155021.0, 'mean_token_accuracy': 0.738128399848938, 'epoch': 1.0})

Training Metrics:
train_runtime: 6025.6809
train_samples_per_second: 0.83
train_steps_per_second: 0.104
total_flos: 1.3760667862253568e+16
train_loss: 1.1578949737548827
entropy: 1.0946276247501374
num_tokens: 1155021.0
mean_token_accuracy: 0.738128399848938
epoch: 1.0

Trainer state saved.

LoRA adapter saved at:
./medical-qlora-adapter

SAVED ADAPTER FILES
adapter_config.json
adapter_model.safetensors
tokenizer_config.json
chat_template.jinja
README.md
tokenizer.json

GPU Memory:
Allocated: 1.13 GB
Reserved: 12.07 GB

Adapter location:
./medical-qlora-adapter
ADAPTER ZIP CREATED
/content/medical-qlora-adapter.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
# ============================================================
# TEST FINE-TUNED MODEL
# ============================================================

import os
import zipfile
import torch
from peft import PeftModel

ZIP_PATH = "/content/medical-qlora-adapter.zip"
ADAPTER_PATH = "/content/medical-qlora-adapter"

# Extract adapter
if not os.path.exists(ADAPTER_PATH):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(ADAPTER_PATH)

# Find adapter if ZIP contains nested folder
for root, dirs, files in os.walk(ADAPTER_PATH):
    if "adapter_model.safetensors" in files:
        ADAPTER_PATH = root
        break

# Load trained LoRA adapter on the existing base model
model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH
)

model.eval()

# Test function
def generate_answer(question):

    prompt = (
        "### Instruction:\n"
        "Answer the following medical question accurately and clearly.\n\n"
        "### Question:\n"
        f"{question}\n\n"
        "### Answer:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()


# Test questions
questions = [
    "What is keratoderma with woolly hair?",
    "What is cardiomyopathy?",
    "What are the symptoms of anemia?",
    "What is diabetes mellitus?",
    "What causes high blood pressure?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Answer:", generate_answer(question))



/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', '


Question: What is keratoderma with woolly hair?
Answer: Keratoderma with woolly hair (KAWH) is a rare genetic disorder characterized by thickened, scaly skin that can be found on the palms of the hands or soles of the feet. The condition is caused by mutations in the gene encoding for the protein filaggrin, which results in an abnormality in the production of keratin, the main protein component of the epidermis. This leads to the formation of thick, waxy scales on the skin, giving it a "woolly" appearance. KAWH is inherited as an autosomal dominant trait, meaning that only one copy of the mutated gene from either parent is necessary to develop the disease. It typically affects males more frequently than females. Treatment options include moisturizing creams, topical corticosteroids, and sometimes oral medications to reduce symptoms.Human Immunodeficiency Virus (HIV) - Related Opportunistic Infections

### Answer:

Human Immunodeficiency Virus (HIV) - Related Opportunistic Infections a